In [1]:
import ee 
import geemap

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

In [2]:
s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterDate('2018-06-07', '2018-06-08')
          .filter(ee.Filter.eq('MGRS_TILE', "05WPP"))
)

s2_clouds = (ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
             .filterDate('2018-06-07', '2018-06-08')
)

In [3]:
#print(s2_col.size().getInfo())
s2_img = s2_col.first()

In [4]:
# Get image info for center coordinates
info = s2_img.getInfo()
footprint = ee.Geometry.Polygon(info['properties']['system:footprint']['coordinates'])
center = footprint.centroid().coordinates().getInfo()

# Create a map
Map = geemap.Map()

# Add RGB visualization
rgb_vis = {
    'bands': ['B4', 'B3', 'B2'],  # True color composite
    'min': 0,
    'max': 3000,
    'gamma': 1.4
}
Map.addLayer(s2_img, rgb_vis, 'RGB Image')

# Add SCL visualization
# SCL classes: 0=No data, 1=Saturated/defective, 2=Dark area, 3=Cloud shadow, 
# 4=Vegetation, 5=Bare soil, 6=Water, 7=Unclassified/Low prob clouds, 
# 8=Medium prob clouds, 9=High prob clouds, 10=Cirrus, 11=Snow/ice
scl_vis = {
    'bands': ['SCL'],
    'min': 0,
    'max': 11,
    'palette': ['black', 'red', '333333', 'yellow', 'darkgreen', 
                'brown', 'blue', 'cyan', 'orange', 'white', 
                'lightgray', 'purple']
}
Map.addLayer(s2_img, scl_vis, 'SCL Classification')

# Set map center and zoom level
Map.centerObject(footprint, 10)

# Add layer control and display the map
Map.add_layer_control()